# Config

In [1]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [ ]:
import pandas as pd
import os
import numpy as np

# Cargar datos

In [16]:
# 1) Cargar datos antiguos
path = "/tmp/ocde/new_data"

#Cargar titulo, keys, etc.
filePATH = os.path.join(path, "old_data.xlsx")
df_data = pd.read_excel(filePATH, usecols=["Código VRID", "Título", "Keywords", "Resumen", "Depto Persona",
                                           "Facultad del Proyecto", "Interdisciplinario", "Transdisciplinario"])


# 2) Textos nuevos
#Pagina 0
filePATH = os.path.join(path, "new_data.xlsx")
df_new0 = pd.read_excel(os.path.join(path, filePATH), usecols=["Código VRID", "Título", "Keywords", "Resumen"], sheet_name=0)
df_new1 = pd.read_excel(os.path.join(path, filePATH), usecols=["Código VRID", "Título", "Keywords", "Resumen"], sheet_name=1)

df_new = pd.concat([df_new0, df_new1])

# 3) Etiquedas Datos OCDE y ODS
#Pagina 2
df_ocde0 = pd.read_excel(os.path.join(path, filePATH), usecols=["Código VRID", "Categoria Disciplina", 
                                             "Sub Area Disciplina", "Tipo Área Disciplina"], sheet_name=2)
#Pagina 3
df_ocde1 =  pd.read_excel(os.path.join(path, filePATH), usecols=["Código VRID", "Categoria Disciplina", 
                                             "Sub Area Disciplina", "Tipo Área Disciplina"], sheet_name=3)

df_labels = pd.concat([df_ocde0, df_ocde1])


# 4) Desafíos país
path = "/tmp/desafios/new_data"
filePATH = os.path.join(path, "Datos_DP.xlsx")
df_labels_desafios = pd.read_excel(filePATH, usecols=["Código VRID", "Desafío País"])

# 5) Lineas de investigación
path = "/tmp/data/"
#Cargar titulo, keys, etc.
filePATH = os.path.join(path, "data_lineas_investigacion.xlsx")
df_data_lineas = pd.read_excel(filePATH, 
                        usecols=["Código VRID", "Título", "Keywords", "Resumen", 
                                 "Línea de investigación"]).drop_duplicates("Código VRID")


In [17]:
df_all = pd.concat([df_data["Código VRID"], df_new["Código VRID"], df_labels["Código VRID"],
                    df_labels_desafios["Código VRID"], df_data_lineas["Código VRID"]])
print(df_all.shape)
df_all = df_all.drop_duplicates()
print("Cantidad total de textos:", df_all.shape)

(5179,)
Cantidad total de textos: (1991,)


# 1) Agregar Textos ODS

In [18]:
df_new["Código VRID"] = df_data["Código VRID"].astype(str)
df_labels["Código VRID"] = df_labels["Código VRID"].astype(str)


# Merge en base a "Código VRID"
df_new_merged = pd.merge(
    df_new,
    df_labels,
    on="Código VRID",       # columna clave
    how="outer"             # inner = solo los que coinciden en ambos
)

#df_new_merged = df_new_merged.drop_duplicates(subset=["Código VRID"])
#save dataframe
print(df_new_merged.shape)


df_new_merged["Tipo Área Disciplina"].value_counts()

(2142, 7)


OCDE    907
ODS      54
Name: Tipo Área Disciplina, dtype: int64

In [27]:
# Filtrar filas donde Tipo Área Disciplina == "ODS" y Título NO es NaN
filtro = (df_new_merged["Tipo Área Disciplina"] == "ODS") & (df_new_merged["Título"].notna())
df_new_merged[filtro].head() 

,Código VRID,Título,Keywords,Resumen,Categoria Disciplina,Sub Area Disciplina,Tipo Área Disciplina


# 2) Agregar area y subarea OCDE

In [28]:
df_data["Código VRID"] = df_data["Código VRID"].astype(str)

# Merge en base a "Código VRID"
df_merged_data = pd.merge(
    df_data,
    df_new_merged,
    on="Código VRID",       # columna clave
    how="outer"            
)

#save dataframe
print(df_merged_data.shape)
df_merged_data["Título"] = df_merged_data["Título_x"].combine_first(df_merged_data["Título_y"])
df_merged_data["Keywords"] = df_merged_data["Keywords_x"].combine_first(df_merged_data["Keywords_y"])
df_merged_data["Resumen"] = df_merged_data["Resumen_x"].combine_first(df_merged_data["Resumen_y"])
df_merged_data = df_merged_data.drop(columns=["Título_x", "Título_y", "Keywords_x", "Keywords_y", "Resumen_x", "Resumen_y"])

df_merged_data.columns

(2142, 14)


Index(['Código VRID', 'Interdisciplinario', 'Transdisciplinario',
       'Facultad del Proyecto', 'Depto Persona', 'Categoria Disciplina',
       'Sub Area Disciplina', 'Tipo Área Disciplina', 'Título', 'Keywords',
       'Resumen'],
      dtype='object')

# 3) Agregar líneas de investigación

In [30]:
df_data_lineas["Código VRID"] = df_data_lineas["Código VRID"].astype(str)

# Merge en base a "Código VRID"
df_merged_lineas = pd.merge(
    df_data_lineas,
    df_merged_data,
    on="Código VRID",       # columna clave
    how="outer"            
)

#save dataframe
df_merged_lineas["Título"] = df_merged_lineas["Título_x"].combine_first(df_merged_lineas["Título_y"])
df_merged_lineas["Keywords"] = df_merged_lineas["Keywords_x"].combine_first(df_merged_lineas["Keywords_y"])
df_merged_lineas["Resumen"] = df_merged_lineas["Resumen_x"].combine_first(df_merged_lineas["Resumen_y"])
df_merged_lineas = df_merged_lineas.drop(columns=["Título_x", "Título_y", "Keywords_x", "Keywords_y", "Resumen_x", "Resumen_y"])
print(df_merged_lineas.shape)
df_merged_lineas.columns

(2142, 12)


Index(['Código VRID', 'Línea de investigación', 'Interdisciplinario',
       'Transdisciplinario', 'Facultad del Proyecto', 'Depto Persona',
       'Categoria Disciplina', 'Sub Area Disciplina', 'Tipo Área Disciplina',
       'Título', 'Keywords', 'Resumen'],
      dtype='object')

# 4) Agregar Desafíos país

In [31]:
# Merge en base a "Código VRID"
df_merged_final = pd.merge(
    df_merged_lineas,
    df_labels_desafios,
    on="Código VRID",       # columna clave
    how="outer"             # inner = solo los que coinciden en ambos
)

#save dataframe
print(df_merged_final.shape)

(2329, 13)


In [32]:
df_merged_final.columns

Index(['Código VRID', 'Línea de investigación', 'Interdisciplinario',
       'Transdisciplinario', 'Facultad del Proyecto', 'Depto Persona',
       'Categoria Disciplina', 'Sub Area Disciplina', 'Tipo Área Disciplina',
       'Título', 'Keywords', 'Resumen', 'Desafío País'],
      dtype='object')

In [36]:
#Save data
path = "/tmp/all_data"
savepath = os.path.join(path, "all_data.csv")
df_merged_final.to_csv(savepath, index=False, encoding="utf-8-sig")

In [ ]:
df_merged_final[df_merged_final["Desafío País"].notna()].drop_duplicates("Código VRID").shape

### Fuera de esto: ###
#Eliminar duplicados
#Eliminar en OCDE los correspondientes a ODS
#Eliminar elementos sin título

(1235, 13)

# Preparación data

In [64]:
#Save data
path = "/tmp/all_data"
filepath = os.path.join(path, "all_data.csv")
df = pd.read_csv(filepath)

## Lectura Interdiciplinario

In [58]:
df = df[df["Interdisciplinario"].notna()].drop_duplicates("Código VRID")
df.shape

(1083, 13)

## Lectura Desafío país

In [60]:
df = df[df["Desafío País"].notna()].drop_duplicates("Código VRID")
df.shape

(1056, 13)

## Lectura OCDE

In [61]:
df = df[df["Tipo Área Disciplina"].notna()].drop_duplicates("Código VRID")
df.shape

(408, 13)

## Lectura líneas de investigación

In [62]:
df = df[df["Línea de investigación"].notna()].drop_duplicates("Código VRID")
df.shape

(51, 13)

# Features dataframe

In [73]:
#Save data
path = "/tmp/all_data"
filepath = os.path.join(path, "all_data.csv")
df = pd.read_csv(filepath)

df = df.drop_duplicates("Código VRID")
df = df.drop(['Título', 'Keywords', 'Resumen', 'Depto Persona', 'Facultad del Proyecto'], axis=1)

In [75]:
#Save data
path = "/tmp/all_data"
savepath = os.path.join(path, "features.csv")
df_merged_final.to_csv(savepath, index=False, encoding="utf-8-sig")

# Prepare data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from utils.dataset import to_serializable
import json

# Función utilizada para seleccionar columna en base a la cual se van a dividir los datos
def get_dataset_to_split(df, feat_col, drop_duplicates_col = "Código VRID"):
    """
    Filtra y selecciona datos de interés desde un DataFrame.

    Inputs:
    - df (pd.DataFrame): DataFrame de entrada con varias columnas.
    - feat_col (str): Nombre de la columna con la característica de interés.
    - drop_duplicates_col (str, opcional): Columna usada para eliminar duplicados
      (por defecto "Código VRID").

    Outputs:
    - pd.DataFrame: DataFrame reducido que contiene solo dos columnas:
      [drop_duplicates_col, feat_col], sin duplicados y sin valores NaN en feat_col.
    """
    df = df[df[feat_col].notna()].drop_duplicates(drop_duplicates_col)
    df = df[[drop_duplicates_col, feat_col]]
    return df

#Funcion  de división de datos
def split_dataset(filepath, ids, labels):
    """
    Divide los datos en train/test, genera folds de validación y guarda índices en JSON.

    Inputs:
    - filepath (str): Ruta donde se guardará el archivo JSON con los índices.
    - ids (array-like): Identificadores de las instancias (ej: códigos VRID).
    - labels (array-like): Etiquetas de las instancias (categorías o clases).

    Outputs:
    - dict: Diccionario con:
        {
            "Test": array de IDs para el conjunto de test,
            "kfolds": lista de arrays de IDs para cada fold de validación
        }
      Además, guarda este diccionario en disco en formato JSON.
    """
    le = LabelEncoder()
    labels = le.fit_transform(labels)

    idx_train, idx_test, y_train, y_test = train_test_split(
        ids,
        labels,
        test_size=0.2,
        random_state=7,
        stratify=labels
    )

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)
    folds = []
    for _, val_pos in skf.split(idx_train, y_train):
        val_ids = idx_train[val_pos]
        folds.append(val_ids)

    print("Test size:", len(idx_test))
    print("Fold 0 - Val size:", len(folds[0]))

    dataset_index = {
        "Test": idx_test,
        "kfolds": folds
    }

    try:
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)
        print("Archivo guardado exitosamente en", filepath)
    except Exception as e:
        print("Error al guardar el archivo:", e)


In [104]:
#Save data
path = "/tmp/all_data"
filepath = os.path.join(path, "features.csv")
df=pd.read_csv(filepath)

feat_col = "Interdisciplinario"
df = get_dataset_to_split(df, feat_col)

#Split data
ids = np.array(df["Código VRID"])
labels = np.array(df["Interdisciplinario"])
savepath = os.path.join(path, "splits/train_test_ids_3folds.json")
split_dataset(savepath, ids, labels)

Test size: 217
Fold 0 - Val size: 289
Archivo guardado exitosamente en /tmp/all_data/splits/train_test_ids_3folds.json
